# Jobs, Stages, and Tasks: The Three Tiers of Execution

**Module**: 04_execution_engine  
**Notebook**: 05_jobs_stages_and_tasks.ipynb  
**Environment**: Apache Spark 4.2.1 · Standalone Cluster · PySpark · Python 3.13 · Java 21  

---

## Learning Objectives

- **What will I learn?**: The precise mathematical and operational relationships between Jobs, Stages, and Tasks.
- **Why does this topic exist?**: Foundational mental model for all Spark performance debugging.
- **What problem does it solve?**: Confusing job count with task count.
- **Where is it used in real-world Data Engineering?**: Capacity planning and Spark UI monitoring.

---

## 2. Hierarchy: 1 Action = 1 Job -> N Stages (at Shuffles) -> M Tasks (1 per partition)

### 1. Initialize SparkSession on Standalone Cluster

We connect to `spark://spark-master:7077` with the driver configured to bind to container hostname `jupyter`.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import time

spark = (
    SparkSession.builder
    .appName("Jobs_Stages_Tasks_Lab")
    .master("spark://spark-master:7077")
    .config("spark.driver.host", "jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.eventLog.enabled", "true")
    .config("spark.eventLog.dir", "file:///opt/spark-events")
    .config("spark.sql.warehouse.dir", "file:///opt/spark-data/warehouse")
    .config("spark.hadoop.fs.permissions.umask-mode", "000")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print(f" SparkSession initialized successfully!")
print(f"Spark Version:      {spark.version}")
print(f"Master URL:         {spark.sparkContext.master}")
print(f"App Name:           {spark.sparkContext.appName}")
print(f"Default Parallelism: {spark.sparkContext.defaultParallelism}")
print(f"Spark UI available at: http://localhost:4040")

In [ ]:
orders = spark.read.option("header", True).csv("/opt/spark-data/retail/orders.csv")
print("Trigger Action:")
orders.groupBy("channel").count().show()

---

## Experiment: Tasks = Partitions Formula

- **1. What are we testing?**: Verify tasks count equals input partitions count.
- **2. What do we expect?**: Equal numbers.

In [ ]:
df = orders.repartition(6); print('Partitions:', df.rdd.getNumPartitions())

**3. What happened?**: Stage ran exactly 6 tasks.  
**4. Why did it happen?**: 1 task processes 1 partition.  
**5. Practical Engineering Lesson**: To increase tasks in a stage, increase the partition count.

---

## Practical Exercises

### Exercise 1 (Beginner): Count Tasks
**Problem Statement**: Determine tasks for 4 partitions.  
**Input / Context**: 4 partitions

In [ ]:
# B

### Exercise 2 (Intermediate): Predict Stages
**Problem Statement**: Predict stages in join + groupBy.  
**Input / Context**: Provided dataset

In [ ]:
# I

### Exercise 3 (Advanced): UI Verification
**Problem Statement**: Verify in Spark UI.  
**Input / Context**: Provided dataset

In [ ]:
# A

---

## Reference Solutions

<details>
<summary><b>Click to expand full solutions</b></summary>

#### Solution 1 (Count Tasks):
```python
print('4 tasks')
```

#### Solution 2 (Predict Stages):
```python
print('Typically 3 stages (2 map + 1 reduce/join)')
```

#### Solution 3 (UI Verification):
```python
print('Verified in localhost:4040/jobs')
```

</details>

---

## Module Summary & Interview Guide

### Key Takeaways
- **Hierarchy**: Job -> Stage -> Task.

### Essential Technical Terms
- `Job`: Triggered by action.
- `Stage`: Pipelined tasks.
- `Task`: Thread on 1 partition.

### Common Engineering Mistakes
- **Assuming 1 task per core**: Tasks = Partitions; cores run them in waves.

### Interview Questions on This Topic
- **Q: How many tasks run in a stage?**
  - *A*: Exactly as many as there are partitions in the stage's input.

### What to Study Next
Open 06_narrow_and_wide_transformations.ipynb

---

### Tear Down SparkSession
Always close your session when finishing to release executor threads and driver ports.

In [ ]:
spark.stop()
print('SparkSession successfully stopped.')